# HGRIA - Hand Gesture Recognition for Interactive Applications
## Local Launch Notebook

```
┌─────────────────────────────────────────────────────────────┐
│                    ARCHITECTURE (LOCAL)                     │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│   Browser (Frontend)     ngrok Tunnel      Local Backend   │
│   ┌──────────────┐      ┌──────────┐     ┌──────────────┐   │
│   │  GitHub Pages │ ←─── │  HTTPS   │ ←── │  Flask +     │   │
│   │  / Vercel    │      │  Tunnel  │     │  MediaPipe   │   │
│   └──────────────┘      └──────────┘     └──────────────┘   │
│                                                 │           │
│                                            <project>/logs/  │
└─────────────────────────────────────────────────────────────┘
```

### Prerequisites
- Python 3.10+ with pip
- ngrok account (free tier works)
- WebRTC-compatible browser (Chrome, Edge, Firefox)
- All dependencies installed (`pip install -r requirements.txt`)

### How it works
1. Backend runs Flask server locally with MediaPipe
2. ngrok creates HTTPS tunnel to expose backend
3. Frontend connects via WebSocket and sends webcam frames
4. Backend processes frames and sends gesture commands back

In [ ]:
# Step 1: Verify dependencies
import numpy as np
import google.protobuf
import mediapipe as mp
import tensorflow as tf

print("NumPy:", np.__version__)
print("Protobuf:", google.protobuf.__version__)
print("MediaPipe:", mp.__version__)
print("TensorFlow:", tf.__version__)
print("MediaPipe OK")
print("TensorFlow OK")

In [ ]:
# Step 2: Set up project root and Python path
import os
import sys
from pathlib import Path

# Notebook lives at <project_root>/notebooks/ — walk up one level
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

project_root_str = str(PROJECT_ROOT)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

print(f"✓ Project root : {PROJECT_ROOT}")

In [ ]:
# Step 3: Configure ngrok authentication
import getpass
import subprocess

subprocess.run(['pip', 'install', '-q', 'pyngrok'], capture_output=True)
from pyngrok import ngrok

print("Enter your ngrok authtoken (from https://dashboard.ngrok.com/auth)")
print("Press Enter to skip (anonymous tunnel may disconnect)")

authtoken = getpass.getpass(prompt='Authtoken: ')

if authtoken:
    ngrok.set_auth_token(authtoken)
    print("✓ ngrok authenticated")
else:
    print("⚠ Anonymous tunnel - connection may be unstable")

In [ ]:
# Step 4: Start ngrok tunnel and display connection info
tunnel = ngrok.connect(5000, "http")
ngrok_url = tunnel.public_url.replace('http://', 'https://')

print("=" * 60)
print("🔗 NGROK TUNNEL READY")
print("=" * 60)
print(f"\nBackend URL: {ngrok_url}")

frontend_script = f'<script>window.HGRIA_BACKEND_URL="{ngrok_url}";</script>'
print(f"\nPaste this in your frontend HTML (before Socket.IO loads):")
print(f"\n{frontend_script}")

frontend_base = "https://qtannguyen-researcher.github.io/HGRIA"
frontend_url = f"{frontend_base}?server={ngrok_url}"

print(f"\nOr open Frontend directly with:")
print(f"\n{frontend_url}")
print("\n" + "=" * 60)

In [ ]:
# Step 5: Patch config for local environment
import json

config_path = str(PROJECT_ROOT / 'config' / 'config.json')

with open(config_path, 'r') as f:
    config = json.load(f)

config['camera']['colab_mode'] = False
config['server']['cors_origins'] = '*'
config['logging']['log_to_file'] = True

log_dir = PROJECT_ROOT / 'logs'
log_dir.mkdir(exist_ok=True)
config['logging']['log_file_path'] = str(log_dir) + '/'

with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print("✓ Config patched:")
print(f"  - colab_mode   : {config['camera']['colab_mode']}")
print(f"  - cors_origins : {config['server']['cors_origins']}")
print(f"  - log_file_path: {config['logging']['log_file_path']}")

In [ ]:
# Step 6: Start the HGRIA Server (blocking)
# This cell will keep running until interrupted (Kernel > Interrupt)
from backend.main import SystemOrchestrator

print("Starting HGRIA Backend Server...")
print(f"Server running at: {ngrok_url}")
print("\nPress Stop button (■) or Kernel > Interrupt to terminate")
print("-" * 40)

orchestrator = SystemOrchestrator(config_path)
orchestrator.start()

## Post-Launch Instructions

### Accessing the Frontend

After the server starts, open your browser and navigate to:

```
https://qtannguyen-researcher.github.io/HGRIA/?server=<NGROK_URL>
```

### If ngrok URL Changes

1. Stop the server (interrupt cell 6)
2. Re-run cells 4, 5, and 6 in sequence
3. Update the frontend with the new URL

### Troubleshooting

| Issue | Solution |
|-------|----------|
| ngrok URL changed | Re-run cells 4, 5, 6 and update frontend |
| Webcam denied | Use keyboard fallback (Arrow keys, Space, P, S) |
| Port 5000 in use | Kill existing process: `lsof -ti:5000 | xargs kill` |

### Keyboard Controls (Fallback)
- Arrow Keys: Move
- Space: Jump
- P: Pause
- S: Speed Boost
- Enter: Confirm